# 01 · Parte 1 — Datos, limpieza y validación

**Responsable:** _(Jose Angel Morales Nuñez)_

**Objetivo:** dejar un dataset limpio y un esquema de validación que las otras dos partes consuman tal cual. Esta parte es la base: sin ella, las partes 2 y 3 no pueden medirse.

Referencia: hallazgos del notebook `00_exploracion.ipynb` (secciones 1, 3, 4, 7 y 8).

Convenciones comunes a las tres partes:

- Rutas relativas a la raíz del repo: `../data/raw/`, `../data/processed/`, `../models/`, `../submissions/`.
- Todo lo que una parte produce para otra se guarda en `../data/processed/<parte>/` en formato parquet, con las columnas documentadas en la sección *Salidas* de cada notebook.
- Identificador de pozo: el prefijo de 8 caracteres del archivo (`000d7d20`). Índice de punto: la fila del CSV horizontal (el mismo que usa `sample_submission`).
- Métrica oficial: RMSE sobre los puntos con índice ≥ PS (Prediction Start), en pies.

## Alcance

1. **Carga unificada**: función que, dado un id de pozo, devuelve el CSV horizontal y el typewell con tipos correctos y el índice del PS.
2. **Limpieza de GR**: tratamiento de NaN del pozo horizontal (huecos de 1-9 ft interpolables; huecos largos marcados con una máscara, no inventados). Decidir y documentar si se suaviza o normaliza el GR y cómo.
3. **Remuestreo del typewell** a una grilla común de TVT (los typewells vienen a 0.25 / 0.5 / 1 ft) y tratamiento de los 13 pozos cuyo typewell no cubre su rango de TVT.
4. **Tabla maestra de pozos**: una fila por pozo con metadatos (n, PS, azimut, coordenadas de cabeza y fin, typewell asignado, formación dominante…).
5. **Esquema de validación**: partición por pozos con criterio espacial (hay vecinos a < 50 ft) en K folds reproducibles. Guardar la asignación pozo → fold.
6. **Función de evaluación**: RMSE por pozo y global sobre el tramo ≥ PS, a partir de un DataFrame de predicciones con el formato acordado.
7. **Formato de predicción**: fijar el esquema que las partes 2 y 3 deben producir y escribir un validador.

## Salidas (contrato con las partes 2 y 3)

| Archivo | Contenido |
|---|---|
| `../data/processed/p1/wells.parquet` | tabla maestra: una fila por pozo (`well`, `n`, `ps`, `azimuth`, `x0`, `y0`, `x_end`, `y_end`, `fold`, …) |
| `../data/processed/p1/horizontal/<well>.parquet` | CSV horizontal limpio: columnas originales + `GR_clean`, `GR_mask` (True = valor interpolado o ausente) |
| `../data/processed/p1/typewell/<well>.parquet` | typewell remuestreado a grilla regular de TVT: `TVT`, `GR`, `Geology` |
| `../data/processed/p1/folds.parquet` | `well`, `fold` |

Formato de predicción que deben respetar las partes 2 y 3:

| Columna | Tipo | Descripción |
|---|---|---|
| `well` | str | id del pozo |
| `idx` | int | fila del CSV horizontal (≥ PS) |
| `tvt_pred` | float | TVT predicho en pies |

Funciones a exponer (copiarlas a un módulo `src/` cuando se estabilicen): `load_well`, `clean_gr`, `resample_typewell`, `make_folds`, `rmse_by_well`, `validate_prediction`.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed/p1")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
train = sorted((RAW / "train").rglob("*"))
test = sorted((RAW / "test").rglob("*"))
print("train:", len(train), "elementos")
for p in train[:8]:
    print("  ", p.relative_to(RAW))
print("test:", len(test), "elementos")
for p in test[:8]:
    print("  ", p.relative_to(RAW))

sub = pd.read_csv(RAW / "sample_submission.csv")
print(sub.shape)
print(sub.head())

In [ ]:
well = "000d7d20"
hz = pd.read_csv(RAW / "train" / f"{well}__horizontal_well.csv")
tw = pd.read_csv(RAW / "train" / f"{well}__typewell.csv")

print("horizontal:", hz.shape)
print(hz.dtypes)
print(hz.head())
print("NaN por columna:")
print(hz.isna().sum())

print("typewell:", tw.shape)
print(tw.dtypes)
print(tw.head())

## 2. Carga unificada

In [ ]:
def load_well(well, split="train"):
    """Devuelve (horizontal, typewell, ps) de un pozo."""
    base = RAW / split
    hz = pd.read_csv(base / f"{well}__horizontal_well.csv")
    tw = pd.read_csv(base / f"{well}__typewell.csv")
    falta = hz["TVT_input"].isna()
    ps = int(falta.idxmax()) if falta.any() else len(hz)
    return hz, tw, ps

In [ ]:
hz, tw, ps = load_well("000d7d20")
print("columnas typewell:", tw.columns.tolist())
print("PS calculado:", ps, "| filas del CSV:", len(hz))
print(hz[["TVT", "TVT_input", "GR"]].isna().sum())

ids = (sub[sub["id"].str.startswith("000d7d20_")]["id"]
       .str.split("_").str[1].astype(int))
print("sample_submission:", ids.min(), "a", ids.max(), "|", len(ids), "filas")

In [ ]:
wells = sorted(p.name[:8] for p in (RAW / "train").glob("*__horizontal_well.csv"))
print(len(wells), "pozos en train")

raros = []
for w in wells:
    hz, tw, ps = load_well(w)
    cola_ok = (hz["TVT_input"].iloc[:ps].notna().all()
               and hz["TVT_input"].iloc[ps:].isna().all())
    if not cola_ok:
        raros.append(w)
print("pozos con TVT_input no contiguo:", len(raros), raros[:5])

hz_t, tw_t, ps_t = load_well("000d7d20", split="test")
print("columnas en test:", hz_t.columns.tolist(), "| PS:", ps_t)

## 3. Limpieza de GR

## 4. Remuestreo del typewell

## 5. Tabla maestra de pozos

## 6. Esquema de validación (folds espaciales)

## 7. Función de evaluación y validador del formato de predicción

## 8. Exportar salidas a `data/processed/p1/`

## Checklist de entrega

- [ ] Las salidas existen para los 773 pozos y se cargan sin errores.
- [ ] `rmse_by_well` reproduce 15.9 ft para la predicción "TVT constante = TVT(PS)" (valor de referencia del notebook 00).
- [ ] Los folds no mezclan pozos vecinos (< 500 ft) entre train y validación.
- [ ] Documentadas las decisiones de limpieza de GR y remuestreo.